# V2 — Generacja RAW: perplexity, capacity, binoculars (Colab)

Uruchamia **wszystkie kombinacje** (poza HumanEval):

- testy: `perplexity`, `capacity`, `binoculars`
- modele: `qwen`, `llama`, `gemma`
- progi: `0.0`, `0.01`, `0.05`, `0.1`

Razem **36 runów** × 4 prompty = szybkie w porównaniu z HumanEval.

Wyniki: `runs/<timestamp>_<model>_<test>_th.../raw_responses.jsonl` na Drive.

Po generacji oceń lokalnie (`run_evaluate.py`) lub na Colab z GPU (`Colab_Evaluate.ipynb` + `requirements-gpu`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Steganography_benchmarks_V2')
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
RUNS_DIR = PROJECT_DIR / 'runs'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)

print('PROJECT_DIR:', PROJECT_DIR)
print('RUNS_DIR:', RUNS_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
import os

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN: ')

In [ ]:
# --- KONFIGURACJA BATCH ---
TESTS = ('perplexity', 'capacity', 'binoculars')
MODELS = ('qwen', 'llama', 'gemma')
THRESHOLDS = (0.0, 0.01, 0.05, 0.1)

TOP_N = 16
MAX_NEW_TOKENS = 512
SKIP_COMPLETED = True      # pomiń runy ze statusem completed w runs/
CONTINUE_ON_ERROR = True   # idź dalej po błędzie pojedynczego runu

# Ogranicz zakres (None = wszystko):
ONLY_TESTS = None          # np. ('perplexity',)
ONLY_MODELS = None       # np. ('qwen',)
ONLY_THRESHOLDS = None     # np. (0.01, 0.05)

import json
import sys
from pathlib import Path

from notebook_runner import run_live


def _th_match(a: float, b: float) -> bool:
    return abs(float(a) - float(b)) < 1e-9


def find_completed_run(runs_dir: Path, test: str, model: str, threshold: float) -> Path | None:
    if not runs_dir.exists():
        return None
    candidates: list[tuple[str, Path]] = []
    for manifest_path in runs_dir.glob('*/manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            continue
        if manifest.get('status') != 'completed':
            continue
        if manifest.get('test') != test:
            continue
        if manifest.get('model_key') != model:
            continue
        if not _th_match(manifest.get('threshold', -1), threshold):
            continue
        candidates.append((manifest.get('updated_at', ''), manifest_path.parent))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def build_cmd(test: str, model: str, threshold: float) -> list[str]:
    return [
        sys.executable,
        'generate_responses.py',
        '--test', test,
        '--model', model,
        '--threshold', str(threshold),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', 'colab',
        '--output-root', 'runs',
    ]


tests = tuple(ONLY_TESTS or TESTS)
models = tuple(ONLY_MODELS or MODELS)
thresholds = tuple(ONLY_THRESHOLDS or THRESHOLDS)

jobs = [(t, m, th) for t in tests for m in models for th in thresholds]
print(f'Plan: {len(jobs)} runów')
print(f'  testy: {tests}')
print(f'  modele: {models}')
print(f'  progi: {thresholds}')

ok, skipped, failed = [], [], []

for index, (test, model, threshold) in enumerate(jobs, start=1):
    label = f'{test} | {model} | th={threshold}'
    print('\n' + '=' * 60)
    print(f'[{index}/{len(jobs)}] {label}')
    print('=' * 60)

    if SKIP_COMPLETED:
        existing = find_completed_run(RUNS_DIR, test, model, threshold)
        if existing is not None:
            print(f'Pomijam — już completed: {existing.name}')
            skipped.append((label, str(existing)))
            continue

    try:
        run_live(build_cmd(test, model, threshold))
        ok.append(label)
    except Exception as exc:
        print(f'BŁĄD: {exc}')
        failed.append((label, str(exc)))
        if not CONTINUE_ON_ERROR:
            raise

print('\n' + '#' * 60)
print('PODSUMOWANIE')
print('#' * 60)
print(f'OK: {len(ok)} | Pominięte: {len(skipped)} | Błędy: {len(failed)}')
if failed:
    print('\nBłędne runy:')
    for label, err in failed:
        print(f'  - {label}: {err}')
    print('\nWznów: ustaw ONLY_* na brakujące i SKIP_COMPLETED=True')

In [ ]:
# Podgląd runów benchmarkowych na Drive
import json
from pathlib import Path

rows = []
for manifest_path in sorted(Path('runs').glob('*/manifest.json')):
    m = json.loads(manifest_path.read_text(encoding='utf-8'))
    if m.get('test') not in ('perplexity', 'capacity', 'binoculars'):
        continue
    rows.append({
        'run': manifest_path.parent.name,
        'test': m.get('test'),
        'model': m.get('model_key'),
        'th': m.get('threshold'),
        'status': m.get('status'),
        'n': f"{m.get('completed_count', '?')}/{m.get('total_tasks', '?')}",
    })

rows.sort(key=lambda r: (r['test'], r['model'], float(r['th'])))
print(f'Runów (bez humaneval): {len(rows)}\n')
for r in rows:
    print(f"{r['status']:12} {r['n']:>5}  th={r['th']:<4}  {r['model']:<6}  {r['test']:<12}  {r['run']}")